In [ ]:
import pandas as pd
import itertools
import statsmodels.api as sm

# =======================
# 1. LOAD DATA
# =======================
df = pd.read_csv('clean_basel_extended.csv')
df['DATE'] = pd.to_datetime(df['DATE'])
df = df.set_index('DATE')

# =======================
# 1B. FILTER DATA 2008–2010
# =======================
df = df.loc['2008-01-01':'2010-12-31']

y = df['BASEL_temp_mean']
exog = df[['BASEL_pressure', 'BASEL_humidity', 'BASEL_cloud_cover']]

# =======================
# 2. PARAMETER GRID (Diperkecil)
# =======================
p = d = q = range(0,2)       # (0,1)
P = D = Q = range(0,2)       # seasonal tetap kecil
seasonal_period = 7          # PALING RINGAN UNTUK DATA DAILY

pdq = list(itertools.product(p,d,q))
seasonal_pdq = list(itertools.product(P,D,Q))

best_aic = float("inf")
best_param = None
best_seasonal_param = None
best_model = None

# =======================
# 3. GRID SEARCH (Ringan)
# =======================
for param in pdq:
    for param_seasonal in seasonal_pdq:
        try:
            model = sm.tsa.SARIMAX(
                y,
                exog=exog,
                order=param,
                seasonal_order=param_seasonal + (seasonal_period,),
                enforce_stationarity=False,
                enforce_invertibility=False,
                simple_differencing=True      # ← bikin jauh lebih ringan
            )
            result = model.fit(disp=False)

            if result.aic < best_aic:
                best_aic = result.aic
                best_param = param
                best_seasonal_param = param_seasonal
                best_model = result

        except:
            continue

print("BEST MODEL FOUND")
print("Order (p,d,q):", best_param)
print("Seasonal (P,D,Q,7):", best_seasonal_param)
print("AIC:", best_aic)

print("\nSummary:")
print(best_model.summary())


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dat

BEST MODEL FOUND
Order (p,d,q): (1, 0, 1)
Seasonal (P,D,Q,7): (0, 1, 1)
AIC: 2886.856785839315

Summary:
                                     SARIMAX Results                                     
Dep. Variable:               DS7.BASEL_temp_mean   No. Observations:                  725
Model:             SARIMAX(1, 0, 1)x(0, 0, 1, 7)   Log Likelihood               -1436.428
Date:                           Sat, 06 Dec 2025   AIC                           2886.857
Time:                                   09:11:11   BIC                           2918.873
Sample:                               01-08-2008   HQIC                          2899.220
                                    - 01-01-2010                                         
Covariance Type:                             opg                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
BAS

/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
import statsmodels.api as sm
from sklearn.metrics import r2_score

# =======================
# 1. FIT FINAL MODEL
# =======================
model = sm.tsa.SARIMAX(
    y,
    exog=exog,
    order=(1,0,1),
    seasonal_order=(0,1,1,7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

result = model.fit(disp=False)

print(result.summary())

# =======================
# 2. R-SQUARE
# =======================
# R-square dihitung dari aktual vs fitted
fitted = result.fittedvalues

# Sesuaikan indeks supaya sama-sama tidak ada NaN
y_clean = y.loc[fitted.index].dropna()
fitted_clean = fitted.loc[y_clean.index].dropna()

r2 = r2_score(y_clean, fitted_clean)
print("R-Squared:", r2)

# =======================
# 3. FORECAST 12 Langkah
# =======================
forecast = result.forecast(
    steps=12,
    exog=exog.tail(12)
)

print("\nForecast 12 Steps Ahead:")
print(forecast)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


                                     SARIMAX Results                                     
Dep. Variable:                   BASEL_temp_mean   No. Observations:                  732
Model:             SARIMAX(1, 0, 1)x(0, 1, 1, 7)   Log Likelihood               -1435.054
Date:                           Sat, 06 Dec 2025   AIC                           2884.108
Time:                                   09:13:38   BIC                           2916.124
Sample:                               01-01-2008   HQIC                          2896.471
                                    - 01-01-2010                                         
Covariance Type:                             opg                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
BASEL_pressure     -152.8723     12.491    -12.238      0.000    -177.355    -128.390
BASEL_humidity       -